In [76]:
# Supress Warnings
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import mplhep as hep
hep.style.use("CMS")
#import seaborn as sns

# Data Science
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets
import xarray as xr

# Geospatial raster data handling
import rioxarray as rxr

# Geospatial data analysis
#import geopandas as gpd

# Geospatial operations
import rasterio
from rasterio import windows  
from rasterio import features  
from rasterio import warp
from rasterio.warp import transform_bounds 
from rasterio.windows import from_bounds 

# Image Processing
from PIL import Image

# Coordinate transformations
from pyproj import Proj, Transformer, CRS

# Feature Engineering
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Machine Learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow import keras
import tf_keras as keras  # Use tf_keras instead of tf.keras
from tf_keras import layers
from tf_keras.models import Model, Sequential, load_model
from tf_keras.layers import Input, Dropout, Dense
from tf_keras.optimizers import Adam
import tensorflow_probability as tfp
import tensorflow_probability.python.layers as tfpl
tfd = tfp.distributions
tfb = tfp.bijectors
from tf_keras.callbacks import CSVLogger, ModelCheckpoint

# Planetary Computer Tools
import pystac_client
import planetary_computer as pc
from pystac.extensions.eo import EOExtension as eo

# Others
import os
from tqdm import tqdm
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)

In [2]:
def combine_two_datasets(dataset1,dataset2):
    '''
    Returns a  vertically concatenated dataset.
    Attributes:
    dataset1 - Dataset 1 to be combined 
    dataset2 - Dataset 2 to be combined
    '''
    
    data = pd.concat([dataset1,dataset2], axis=1)
    return data

In [3]:
# Extracts satellite band values from a GeoTIFF based on coordinates from a csv file and returns them in a DataFrame.

def map_satellite_data(tiff_path, csv_path):
    
    # Load the GeoTIFF data
    data = rxr.open_rasterio(tiff_path)
    tiff_crs = data.rio.crs

    # Read the Excel file using pandas
    df = pd.read_csv(csv_path)
    latitudes = df['Latitude'].values
    longitudes = df['Longitude'].values

    # 3. Convert lat/long to the GeoTIFF's CRS
    # Create a Proj object for EPSG:4326 (WGS84 - lat/long) and the GeoTIFF's CRS
    proj_wgs84 = Proj(init='epsg:4326')  # EPSG:4326 is the common lat/long CRS
    proj_tiff = Proj(tiff_crs)
    
    # Create a transformer object
    transformer = Transformer.from_proj(proj_wgs84, proj_tiff)

    vals = [[] for _ in range(n_inputs)]


# Iterate over the latitudes and longitudes, and extract the corresponding band values
    for lat, lon in tqdm(zip(latitudes, longitudes), total=len(latitudes), desc="Mapping values"):
    # Assuming the correct dimensions are 'y' and 'x' (replace these with actual names from data.coords)
    
        for i in range(n_inputs):
            tmp = data.sel(x=lon, y=lat, band=i, method='nearest').values
            vals[i].append(tmp)

    # Create a DataFrame with the band values
    # Create a DataFrame to store the band values
    df = pd.DataFrame()
    for i in range(n_inputs):
        df[names[i]] = vals[i]
    
    return df


In [4]:
# Constants
n_inputs = 28
names = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12", "ndwi", "ndvi", "ndbi", 
        "B01 (pooled)", "B02 (pooled)", "B03 (pooled)", "B04 (pooled)", "B05 (pooled)", "B06 (pooled)", "B07 (pooled)", "B08 (pooled)", "B8A (pooled)", "B11 (pooled)", "B12 (pooled)", "ndwi (pooled)", "ndvi (pooled)", "ndbi (pooled)",
        "building_count_0-25ft", "building_impact_0-25ft", "building_count_25-50ft", "building_impact_25-50ft", "building_count_50-100ft", "building_impact_50-100ft", "building_count_100-200ft", "building_impact_100-200ft", "building_count_200-500ft", "building_impact_200-500ft",
        "tree_count_0-25ft", "tree_impact_0-25ft", "tree_count_25-50ft", "tree_impact_25-50ft", "tree_count_50-100ft", "tree_impact_50-100ft", "tree_count_100-200ft", "tree_impact_100-200ft", "tree_count_200-500ft", "tree_impact_200-500ft"]
UhiData_FileName = "Data/Training_data_uhi_index.csv"
satellite_data =  "Data/S2_median_fullBands_indeces.tiff"
tree_building_uhi_data = "Data/LATEST_combined_tree_building_uhi_features.csv"
test_size = 0.3

# Open the GeoTIFF file
tiff_path = "Data/S2_median_fullBands_indeces.tiff"

# Read the bands from the GeoTIFF file
with rasterio.open(tiff_path) as src1:
    cols = [src1.read(i) for i in range(1, n_inputs+1)]

In [5]:
# Would be faster in a df
feature_data = map_satellite_data(satellite_data, UhiData_FileName)

Mapping values: 100%|██████████| 11229/11229 [07:50<00:00, 23.85it/s]


In [7]:
# Redo columns
feature_data["ndwi"] = (feature_data["B03"] - feature_data["B08"])/(feature_data["B03"] + feature_data["B08"]) #Normalized Difference Water Index
feature_data["ndbi"] = (feature_data["B11"] - feature_data["B08"])/(feature_data["B11"] + feature_data["B08"]) # Normalized Difference Buildup Index
feature_data["ndvi"] = (feature_data["B08"] - feature_data["B04"])/(feature_data["B08"] + feature_data["B04"]) #Normalized Difference Vegetation Index
feature_data["ndwi (pooled)"] = (feature_data["B03 (pooled)"] - feature_data["B08 (pooled)"])/(feature_data["B03 (pooled)"] + feature_data["B08 (pooled)"]) #Normalized Difference Water Index
feature_data["ndbi (pooled)"] = (feature_data["B11 (pooled)"] - feature_data["B08 (pooled)"])/(feature_data["B11 (pooled)"] + feature_data["B08 (pooled)"]) # Normalized Difference Buildup Index
feature_data["ndvi (pooled)"] = (feature_data["B08 (pooled)"] - feature_data["B04 (pooled)"])/(feature_data["B08 (pooled)"] + feature_data["B04 (pooled)"]) #Normalized Difference Vegetation Index

feature_data['ndvi'] = feature_data['ndvi'].replace([np.inf, -np.inf], np.nan) 
feature_data['ndwi'] = feature_data['ndwi'].replace([np.inf, -np.inf], np.nan) 
feature_data['ndbi'] = feature_data['ndbi'].replace([np.inf, -np.inf], np.nan)
feature_data['ndvi (pooled)'] = feature_data['ndvi (pooled)'].replace([np.inf, -np.inf], np.nan) 
feature_data['ndwi (pooled)'] = feature_data['ndwi (pooled)'].replace([np.inf, -np.inf], np.nan) 
feature_data['ndbi (pooled)'] = feature_data['ndbi (pooled)'].replace([np.inf, -np.inf], np.nan) 

In [8]:
# Combining ground data and feature data into a single dataset.
ground_df = pd.read_csv(UhiData_FileName)
tree_building_df = pd.read_csv(tree_building_uhi_data)
uhi_data = combine_two_datasets(ground_df,feature_data)
uhi_data = combine_two_datasets(uhi_data, tree_building_df)

In [9]:
# Remove duplicate rows from the DataFrame based on specified columns and keep the first occurrence
columns_to_check = ['B01','B04','B06','B08','ndvi']
for col in columns_to_check:
    # Check if the value is a numpy array and has more than one dimension
    uhi_data[col] = uhi_data[col].apply(lambda x: tuple(x) if isinstance(x, np.ndarray) and x.ndim > 0 else x)

# Now remove duplicates
uhi_data = uhi_data.drop_duplicates(subset=columns_to_check, keep='first')
uhi_data.head()

,Longitude,Latitude,datetime,UHI Index,B01,B02,B03,B04,B05,B06,...,tree_count_50-100ft,tree_impact_50-100ft,tree_count_100-200ft,tree_impact_100-200ft,tree_count_200-500ft,tree_impact_200-500ft,closest_tree_distance,closest_tree_dbh,longitude,latitude
0,-73.909167,40.813107,24-07-2021 15:53,1.030289,841.5,841.5,1053.0,1155.0,1206.0,1481.5,...,2.0,25.845166,13.0,382.989385,51.0,694.608498,40.498933,5.0,-73.909167,40.813107
2,-73.909215,40.812978,24-07-2021 15:53,1.023798,841.5,841.5,646.0,823.0,777.0,1130.5,...,4.0,71.716201,9.0,286.517317,53.0,731.532544,44.856958,7.0,-73.909215,40.812978
3,-73.909242,40.812908,24-07-2021 15:53,1.023798,841.5,841.5,625.0,766.0,741.5,1130.5,...,2.0,51.704779,8.0,249.112379,52.0,671.714205,18.688324,7.0,-73.909242,40.812908
4,-73.909257,40.812845,24-07-2021 15:53,1.021634,841.5,841.5,659.5,763.0,708.5,1077.5,...,1.0,10.949187,6.0,174.116477,52.0,723.484668,9.012552,7.0,-73.909257,40.812845
6,-73.909312,40.812710,24-07-2021 15:53,1.015143,841.5,841.5,551.5,768.5,659.0,1077.5,...,1.0,12.113698,3.0,32.561605,54.0,748.122336,35.179383,6.0,-73.909312,40.812710


In [10]:
# Load data, normalize distributions
model_data = uhi_data[names + ['UHI Index']]
md_data_std = model_data.apply(lambda x: (x - x.mean()) / x.std(), axis=0) 
md_data_std =md_data_std.apply(lambda x: pd.to_numeric(x, errors = 'coerce'), axis=0) 

In [11]:
# Divide data into training and testing
X = md_data_std.drop(columns=['UHI Index']).values
y = md_data_std['UHI Index'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size,random_state=123)
print(X_train.shape) 
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(5848, 48)
(5848, 2)
(2507, 48)
(2507, 2)


In [45]:
def test_model(model, X_test, y_test, csv_name, model_name):
    
    # predict; get mse, r2
    y_test = np.array([val[0] for val in y_test])
    y_pred = model.predict(X_test).flatten()
    print(f"MSE: {mean_squared_error(y_test, y_pred)}")
    print(f"r2: {r2_score(y_test, y_pred)}")

    # plotting loss curve
    log = pd.read_csv(csv_name)
    training_loss = log["loss"]
    validation_loss = log["val_loss"]
    plt.plot(np.arange(1, len(training_loss) + 1), training_loss, label="Training")
    plt.plot(np.arange(1, len(validation_loss) + 1), validation_loss, label="Validation")
    plt.legend(loc="upper right")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.savefig(f"../Plots/{model_name}_loss")
    plt.clf()

    # plotting test vs pred dist
    plt.hist(
        y_test,
        bins=100,
        label='Actual',
        density=1,
        histtype="step",
    )
    plt.hist(
        y_pred,
        bins=100,
        label='Predicted',
        density=1,
        histtype="step",
    )
    plt.legend(loc="upper right")
    plt.xlabel("UHI Index")
    plt.ylabel("Frequency")
    plt.xlim(-3, 3)
    plt.savefig(f"../Plots/{model_name}_dist")
    plt.clf()
    
    # scatter plot of test vs pred
    plt.scatter(y_test, y_pred, s = 25, alpha=0.3)
    plt.xlabel("Actual UHI")
    plt.ylabel("Predicted UHI")
    plt.xlim(-3, 3)
    plt.ylim(-3, 3)
    plt.savefig(f"../Plots/{model_name}_scatter")
    plt.clf()


In [ ]:
# Standard Model
model1 = Sequential([
    Dense(64, activation = 'relu', input_shape=(48,) ),
    Dropout(0.1),
    Dense(64, activation = 'relu'),
    Dropout(0.1),
    Dense(64, activation = 'relu'),
    Dropout(0.1),
    Dense(1, activation = 'linear'),
])
model1.compile(optimizer=Adam(learning_rate=0.001) , loss = "MSE" )
mc = ModelCheckpoint("Model_Stats/NN_model", save_best_only=True)
log = CSVLogger(f"Model_Stats/NN_training.log", append=True)
model1.fit(X_train, y_train, epochs=100, validation_split=0.1, callbacks=[mc, log], verbose=1)

Epoch 1/100
161/165 [============================>.] - ETA: 0s - loss: 0.8579INFO:tensorflow:Assets written to: Model_Stats/NN_model/assets


INFO:tensorflow:Assets written to: Model_Stats/NN_model/assets


165/165 [==============================] - 5s 18ms/step - loss: 0.8523 - val_loss: 0.7993
Epoch 2/100
151/165 [==========================>...] - ETA: 0s - loss: 0.7574INFO:tensorflow:Assets written to: Model_Stats/NN_model/assets


INFO:tensorflow:Assets written to: Model_Stats/NN_model/assets


165/165 [==============================] - 2s 11ms/step - loss: 0.7507 - val_loss: 0.7161
Epoch 3/100
148/165 [=========================>....] - ETA: 0s - loss: 0.7035INFO:tensorflow:Assets written to: Model_Stats/NN_model/assets


...

165/165 [==============================] - 2s 14ms/step - loss: 0.2298 - val_loss: 0.3342
Epoch 80/100
156/165 [===========================>..] - ETA: 0s - loss: 0.2325INFO:tensorflow:Assets written to: Model_Stats/NN_model/assets


INFO:tensorflow:Assets written to: Model_Stats/NN_model/assets


165/165 [==============================] - 2s 14ms/step - loss: 0.2354 - val_loss: 0.3274
Epoch 81/100
165/165 [==============================] - 1s 4ms/step - loss: 0.2330 - val_loss: 0.3394
Epoch 82/100
165/165 [==============================] - 1s 3ms/step - loss: 0.2319 - val_loss: 0.3449
Epoch 83/100
165/165 [==============================] - 1s 3ms/step - loss: 0.2345 - val_loss: 0.3344
Epoch 84/100
165/165 [==============================] - 1s 4ms/step - loss: 0.2314 - val_loss: 0.3494
Epoch 85/100
165/165 [==============================] - 1s 3ms/step - loss: 0.2307 - val_loss: 0.3390
Epoch 86/100
165/165 [==============================] - 1s 3ms/step - loss: 0.2320 - val_loss: 0.3333
Epoch 87/100
165/165 [==============================] - 1s 3ms/step - loss: 0.2331 - val_loss: 0.3330
Epoch 88/100
165/165 [==============================] - 1s 3ms/step - loss: 0.2272 - val_loss: 0.3450
Epoch 89/100
165/165 [==============================] - 1s 5ms/step - loss: 0.2236 - val_loss:

INFO:tensorflow:Assets written to: Model_Stats/NN_model/assets


165/165 [==============================] - 2s 14ms/step - loss: 0.2221 - val_loss: 0.3235
Epoch 96/100
165/165 [==============================] - 1s 4ms/step - loss: 0.2231 - val_loss: 0.3415
Epoch 97/100
165/165 [==============================] - 0s 3ms/step - loss: 0.2159 - val_loss: 0.3337
Epoch 98/100
165/165 [==============================] - 0s 3ms/step - loss: 0.2198 - val_loss: 0.3317
Epoch 99/100
165/165 [==============================] - 0s 2ms/step - loss: 0.2150 - val_loss: 0.3289
Epoch 100/100
165/165 [==============================] - 0s 3ms/step - loss: 0.2139 - val_loss: 0.3392


In [52]:
test_model(load_model("Model_Stats/NN_model"), X_test=X_test, y_test=y_test, csv_name = "Model_Stats/NN_training.log", model_name = "NN")

79/79 [==============================] - 1s 2ms/step
MSE: 0.3589788618190154
r2: 0.6416329041493051


<Figure size 1000x1000 with 0 Axes>

In [ ]:
# Fixed prior: standard normal, wrapped in Independent
'''def prior(kernel_size, bias_size=0, dtype=None):
    n = kernel_size + bias_size
    return Sequential([
        tfpl.DistributionLambda(
            lambda t: tfd.Independent(
                tfd.Normal(loc=tf.zeros(n), scale=1.0),
                #tfd.Laplace(loc=tf.zeros(n), scale=1.0),
                #tfd.StudentT(df=3.0, loc=tf.zeros(n), scale=1.0),
                reinterpreted_batch_ndims=1)
        )
    ])''' # Independent, so replaced with dependent prior

def prior(kernel_size, bias_size=0, dtype=None):
    n = kernel_size + bias_size
    # Use a MultivariateNormalDiag with a shared trainable covariance or set it manually.
    loc = tf.zeros(n)
    scale_tril = tf.linalg.cholesky(tf.eye(n))  # Identity = uncorrelated, but you can change this
    return Sequential([
        tfpl.DistributionLambda(
            lambda t: tfd.MultivariateNormalTriL(loc=loc, scale_tril=scale_tril)
        )
    ])

# Posterior: learnable multivariate Gaussian
def posterior(kernel_size, bias_size=0, dtype=None):
    n = kernel_size + bias_size
    return Sequential([
        tfpl.VariableLayer(tfp.layers.MultivariateNormalTriL.params_size(n), dtype=dtype),
        tfpl.MultivariateNormalTriL(n)
    ])

In [ ]:
# Bayesian Model
kl_weight = 1 / X_train.shape[0]  # or start with 1e-4
inputs = Input(shape=(48,))
x = tfp.layers.DenseVariational(
    units=16,
    make_prior_fn=prior,
    make_posterior_fn=posterior,
    kl_weight = kl_weight,
    activation='sigmoid')(inputs)
outputs = Dense(1, activation='linear')(x)

# wider/deeper network still leads to posterior collapse
'''x = tfp.layers.DenseVariational(
    units=64,
    make_prior_fn=prior,
    make_posterior_fn=posterior,
    kl_weight=kl_weight,
    activation='relu')(inputs)
x = tfp.layers.DenseVariational(
    units=32,
    make_prior_fn=prior,
    make_posterior_fn=posterior,
    kl_weight=kl_weight,
    activation='relu')(x)
outputs = Dense(1, activation='linear')(x)'''

model2 = Model(inputs=inputs, outputs=outputs)
model2.compile(optimizer=Adam(0.001), loss="mse")
mc = ModelCheckpoint("Model_Stats/BNN_1_model", save_best_only=True)
log = CSVLogger(f"Model_Stats/BNN_1_training.log", append=True)
model2.fit(X_train, y_train, epochs=20, validation_split=0.1, callbacks=[mc, log], verbose=0)

Epoch 1/20
163/165 [============================>.] - ETA: 0s - loss: 1.3223WARNING:tensorflow:`_` is not a valid node name. Accepted names conform to Regex /re.compile('^[A-Za-z0-9.][A-Za-z0-9_.\\\\/>-]*$')/


INFO:tensorflow:Assets written to: Model_Stats/BNN_1_model/assets


INFO:tensorflow:Assets written to: Model_Stats/BNN_1_model/assets


165/165 [==============================] - 10s 48ms/step - loss: 1.3221 - val_loss: 1.2036
Epoch 2/20
165/165 [==============================] - ETA: 0s - loss: 1.1363WARNING:tensorflow:`_` is not a valid node name. Accepted names conform to Regex /re.compile('^[A-Za-z0-9.][A-Za-z0-9_.\\\\/>-]*$')/


INFO:tensorflow:Assets written to: Model_Stats/BNN_1_model/assets


...

INFO:tensorflow:Assets written to: Model_Stats/BNN_1_model/assets


165/165 [==============================] - 8s 46ms/step - loss: 0.9094 - val_loss: 0.9315
Epoch 13/20
162/165 [============================>.] - ETA: 0s - loss: 0.9103WARNING:tensorflow:`_` is not a valid node name. Accepted names conform to Regex /re.compile('^[A-Za-z0-9.][A-Za-z0-9_.\\\\/>-]*$')/


INFO:tensorflow:Assets written to: Model_Stats/BNN_1_model/assets


INFO:tensorflow:Assets written to: Model_Stats/BNN_1_model/assets


165/165 [==============================] - 7s 45ms/step - loss: 0.9105 - val_loss: 0.9095
Epoch 14/20
165/165 [==============================] - 2s 13ms/step - loss: 0.9009 - val_loss: 0.9471
Epoch 15/20
165/165 [==============================] - 2s 14ms/step - loss: 0.9129 - val_loss: 0.9337
Epoch 16/20
165/165 [==============================] - 2s 11ms/step - loss: 0.8985 - val_loss: 0.9375
Epoch 17/20
165/165 [==============================] - 2s 11ms/step - loss: 0.9066 - val_loss: 0.9443
Epoch 18/20
165/165 [==============================] - 2s 13ms/step - loss: 0.9052 - val_loss: 0.9316
Epoch 19/20
165/165 [==============================] - 2s 10ms/step - loss: 0.8978 - val_loss: 0.9495
Epoch 20/20
165/165 [==============================] - 2s 10ms/step - loss: 0.9038 - val_loss: 0.9263


In [63]:
test_model(load_model("Model_Stats/BNN_1_model"), X_test=X_test, y_test=y_test, csv_name = "Model_Stats/BNN_1_training.log", model_name = "BNN_1")

79/79 [==============================] - 1s 3ms/step
MSE: 0.8859031452068781
r2: 0.11560659659997763


<Figure size 1000x1000 with 0 Axes>

In [71]:
# Now try pretraining BNN
model3_pre = Sequential([
    Dense(16, activation = 'sigmoid', input_shape=(48,) ),
    Dense(1, activation = 'linear'),
])
model3_pre.compile(optimizer=Adam(learning_rate=0.001) , loss = "MSE" )
mc = ModelCheckpoint("Model_Stats/NN_pretrained_model", save_best_only=True)
log = CSVLogger(f"Model_Stats/NN_pretrained_training.log", append=True)
model3_pre.fit(X_train, y_train, epochs=100, validation_split=0.1, callbacks=[mc, log], verbose=0)

INFO:tensorflow:Assets written to: Model_Stats/NN_pretrained_model/assets


INFO:tensorflow:Assets written to: Model_Stats/NN_pretrained_model/assets


INFO:tensorflow:Assets written to: Model_Stats/NN_pretrained_model/assets


INFO:tensorflow:Assets written to: Model_Stats/NN_pretrained_model/assets


INFO:tensorflow:Assets written to: Model_Stats/NN_pretrained_model/assets


...

INFO:tensorflow:Assets written to: Model_Stats/NN_pretrained_model/assets


INFO:tensorflow:Assets written to: Model_Stats/NN_pretrained_model/assets


INFO:tensorflow:Assets written to: Model_Stats/NN_pretrained_model/assets


INFO:tensorflow:Assets written to: Model_Stats/NN_pretrained_model/assets


INFO:tensorflow:Assets written to: Model_Stats/NN_pretrained_model/assets


In [72]:
test_model(load_model("Model_Stats/NN_pretrained_model"), X_test=X_test, y_test=y_test, csv_name = "Model_Stats/NN_pretrained_training.log", model_name = "NN_pretrained")

79/79 [==============================] - 0s 2ms/step
MSE: 0.5798289502887607
r2: 0.42115918482715176


<Figure size 1000x1000 with 0 Axes>

In [ ]:
def pretrained_prior(pretrained_model):
    def prior_fn(kernel_size, bias_size=0, dtype=None):
        n = kernel_size + bias_size
        prior_weights = pretrained_model.get_weights()  # This gives a list of numpy arrays

        # manually extract matching weights for each layer
        # assume layer order matches
        weight_index = prior_fn.layer_index
        prior_mean = tf.convert_to_tensor(
            prior_weights[weight_index].flatten().tolist() + 
            prior_weights[weight_index + 1].flatten().tolist(), dtype=dtype)
        prior_fn.layer_index += 2  # Advance to next layer’s weights

        # Define fixed Gaussian prior centered on pretrained weights
        return Sequential([
            tfpl.DistributionLambda(lambda t: tfd.Independent(
                tfd.Normal(loc=prior_mean, scale=0.05),  # small variance
                reinterpreted_batch_ndims=1))
        ])
    prior_fn.layer_index = 0
    return prior_fn

prior = pretrained_prior(load_model("Model_Stats/NN_pretrained_model"))

def pretrained_posterior(pretrained_model):
    def posterior_fn(kernel_size, bias_size=0, dtype=None):
        n = kernel_size + bias_size
        posterior_weights = pretrained_model.get_weights()

        # Layer index used to extract weights layer-by-layer
        weight_index = posterior_fn.layer_index
        posterior_mean = tf.convert_to_tensor(
            posterior_weights[weight_index].flatten().tolist() + 
            posterior_weights[weight_index + 1].flatten().tolist(), dtype=dtype)
        posterior_fn.layer_index += 2  # move to next layer

        # Default initialization for scale (lower-triangular Cholesky factor)
        def initializer_fn(shape, dtype=None):
            # Mean + unconstrained scale (init to identity)
            loc = posterior_mean
            tril_elements = tf.zeros(tfpl.MultivariateNormalTriL.params_size(n) - n, dtype=dtype)
            return tf.concat([loc, tril_elements], axis=0)

        return Sequential([
            tfpl.VariableLayer(tfp.layers.MultivariateNormalTriL.params_size(n),
                               dtype=dtype,
                               initializer=initializer_fn),
            tfpl.MultivariateNormalTriL(n)
        ])

    posterior_fn.layer_index = 0
    return posterior_fn


In [87]:
# Pretrained bayesian Model
prior = pretrained_prior(load_model("Model_Stats/NN_pretrained_model"))
posterior = pretrained_posterior(load_model("Model_Stats/NN_pretrained_model"))
kl_weight = 1 / X_train.shape[0]  # or start with 1e-4
inputs = Input(shape=(48,))
x = tfp.layers.DenseVariational(
    units=16,
    make_prior_fn=prior,
    make_posterior_fn=posterior,
    kl_weight = kl_weight,
    activation='sigmoid')(inputs)
outputs = Dense(1, activation='linear')(x)

model3 = Model(inputs=inputs, outputs=outputs)
model3.compile(optimizer=Adam(0.001), loss="mse")
mc = ModelCheckpoint("Model_Stats/BNN_after_pretraining_model", save_best_only=True)
log = CSVLogger(f"Model_Stats/BNN_after_pretraining_training.log", append=True)
model3.fit(X_train, y_train, epochs=20, validation_split=0.1, callbacks=[mc, log], verbose=1)

Epoch 1/20
165/165 [==============================] - 9s 42ms/step - loss: 13.8666 - val_loss: 13.3210
Epoch 2/20
165/165 [==============================] - 8s 48ms/step - loss: 13.1254 - val_loss: 12.9981
Epoch 3/20
165/165 [==============================] - 7s 43ms/step - loss: 12.5093 - val_loss: 12.3197
Epoch 4/20
165/165 [==============================] - 5s 32ms/step - loss: 11.8176 - val_loss: 11.4482
Epoch 5/20
165/165 [==============================] - 5s 32ms/step - loss: 11.0628 - val_loss: 10.6704
Epoch 6/20
165/165 [==============================] - 5s 31ms/step - loss: 10.2657 - val_loss: 9.9735
Epoch 7/20
165/165 [==============================] - 7s 40ms/step - loss: 9.6448 - val_loss: 9.2523
Epoch 8/20
165/165 [==============================] - 5s 32ms/step - loss: 8.8814 - val_loss: 8.4329
Epoch 9/20
165/165 [==============================] - 5s 30ms/step - loss: 8.3237 - val_loss: 7.8318
Epoch 10/20
165/165 [==============================] - 5s 31ms/step - loss: 7.70

In [88]:
test_model(load_model("Model_Stats/BNN_after_pretraining_model"), X_test=X_test, y_test=y_test, csv_name = "Model_Stats/BNN_after_pretraining_training.log", model_name = "BNN_after_pretraining")

79/79 [==============================] - 0s 3ms/step
MSE: 0.8453139121400978
r2: 0.15612665815246685


<Figure size 1000x1000 with 0 Axes>

In [105]:
# Prior function: Returns a callable that returns a Normal distribution with batch shape
def prior(kernel_size, bias_size=0, dtype=None):
    def prior_fn(shape): # 'shape' will be a TensorShape of (kernel_size + bias_size,)
        return tfd.Normal(loc=tf.zeros(shape, dtype=dtype), scale=1.0)
    return prior_fn

# Posterior function: Returns a callable that returns a Normal distribution with batch shape
def posterior(kernel_size, bias_size=0, dtype=None):
    def posterior_fn(shape): # 'shape' will be a TensorShape of (kernel_size + bias_size,)
        return tfd.Normal(
            loc=tf.Variable(lambda: tf.random.normal(shape, stddev=0.1, dtype=dtype)),
            scale=tf.math.softplus(tf.Variable(lambda: tf.random.normal(shape, stddev=0.1, dtype=dtype)))
        )
    return posterior_fn

# Define likelihood function (Gaussian)
def neg_log_likelihood(y_true, y_pred_dist):
    return -tf.reduce_mean(y_pred_dist.log_prob(y_true))

# Wrap output as a Normal distribution
def model_output_distribution(y_pred_mean):
    return tfd.Normal(loc=y_pred_mean, scale=1.0)

In [ ]:
print("TensorFlow version:", tf.__version__)
print("TensorFlow Probability version:", tfp.__version__)

# Build Bayesian Regression model with DenseVariational layers
inputs = Input(shape=(48,))

# First Dense Variational layer
x = tfpl.DenseVariational(
    units=16,
    make_prior_fn=prior,
    make_posterior_fn=posterior,
    kl_weight=1 / X_train.shape[0],
    activation='tanh'
)(inputs)

# Second Dense Variational layer for regression output
outputs = tfpl.DenseVariational(
    units=1,
    make_prior_fn=prior,
    make_posterior_fn=posterior,
    kl_weight=1 / X_train.shape[0]
)(x)

# Wrap the output of the second DenseVariational layer as a Normal distribution
distribution_output = tfpl.DistributionLambda(
    lambda t: tfd.Normal(loc=t, scale=1.0)
)(outputs)

# Final regression model
regression_model = Model(inputs=inputs, outputs=distribution_output)

# Compile the model
regression_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss=neg_log_likelihood
)

# Train the model
log = CSVLogger(f"Model_Stats/regression_training.log", append=True)
regression_model.fit(X_train, y_train, epochs=20, validation_split=0.1, callbacks=[log])

TensorFlow version: 2.16.2
TensorFlow Probability version: 0.24.0


TypeError: Exception encountered when calling layer "dense_variational_51" (type DenseVariational).

<tf.Tensor 'Placeholder:0' shape=(None, 48) dtype=float32> is out of scope and cannot be used here. Use return values, explicit Python locals or TensorFlow collections to access it.
Please see https://www.tensorflow.org/guide/function#all_outputs_of_a_tffunction_must_be_return_values for more information.

<tf.Tensor 'Placeholder:0' shape=(None, 48) dtype=float32> was defined here:
    File "/Users/aji/anaconda3/envs/UHI/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    File "/Users/aji/anaconda3/envs/UHI/lib/python3.10/runpy.py", line 86, in _run_code
    File "/Users/aji/anaconda3/envs/UHI/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    File "/Users/aji/anaconda3/envs/UHI/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    File "/Users/aji/anaconda3/envs/UHI/lib/python3.10/site-packages/ipykernel/k

In [ ]:
test_model(regression_model, X_test=X_test, y_test=y_test)
log = pd.read_csv(f"Model_Stats/regression_training.log")
training_loss = log["loss"]
validation_loss = log["val_loss"]
plt.plot(np.arange(1, len(training_loss) + 1), training_loss, label="Training")
plt.plot(np.arange(1, len(validation_loss) + 1), validation_loss, label="Validation")
plt.legend(loc="upper right")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

In [ ]:
# Predict distribution over outputs
pred_dist = regression_model(X_test)
pred_means = pred_dist.mean().numpy()
pred_stddevs = pred_dist.stddev().numpy()

num_samples = 1000

# Store predictions
predictions = []

for _ in range(num_samples):
    pred_dist = regression_model(X_test)    # This samples new weights each time
    sample = pred_dist.sample().numpy()    # Sample a value from the output distribution
    predictions.append(sample[0][0])       # Assuming scalar regression output

# Now predictions is a list of 1000 sampled outputs
predictions = np.array(predictions)

# You can now compute mean, std, percentiles, etc.
predictive_mean = np.mean(predictions)
predictive_std = np.std(predictions)